# Full pipeline — integration test (notebook 07)

All modules active, nothing silenced. Upload any image → see every module's
evidence → read the fused verdict. This is the demo notebook.

**Modules active:**
- **M1** Provenance (C2PA, IPTC, gen-params, camera EXIF)
- **M2** Watermarks (DWT-DCT, TrustMark P/Q/B, Stable Signature BZH)
- **M4** Classifier (DINOv2 + attention-pooling head, SDv1.4-trained)
- **M5** Web provenance (reverse search + page-context analysis)

M3 (FFT forensics) demoted to note-only after false-positiving on real photos.

Convention: commit WITH outputs.

In [ ]:
# ── Setup ──────────────────────────────────────────────────────
!apt-get -qq install -y libimage-exiftool-perl > /dev/null
!git clone -q https://github.com/Waranika/AI-image-Checkers.git 2>/dev/null || echo "already cloned"
%cd /content/AI-image-Checkers
%pip install -q -e .
%pip install -q --no-deps trustmark 2>/dev/null || echo "trustmark unavailable"
%pip install -q transformers 2>/dev/null || echo "transformers unavailable"

import os, sys
sys.path.insert(0, "/content/AI-image-Checkers")

from google.colab import drive, userdata
drive.mount('/content/drive')

# API key for M5 (stored in Colab Secrets, never in the notebook)
try:
    os.environ["GOOGLE_CLOUD_API_KEY"] = userdata.get("GOOGLE_CLOUD_API_KEY")
    print("M5 API key: loaded ✓")
except Exception:
    print("M5 API key: not found (M5 web search will be skipped)")

# Detector checkpoint from Drive
from pathlib import Path
CKPT = "/content/drive/MyDrive/ai_image_id/runs/cb68637/head.pt"
print(f"M4 checkpoint: {'loaded ✓' if Path(CKPT).exists() else '⚠ not found'}")

import torch
print(f"torch: {torch.__version__} | cuda: {torch.cuda.is_available()}")

In [ ]:
from ai_image_id.main import analyze_image
from ai_image_id.web_provenance import trace_provenance
from ai_image_id.schema import Verdict

def full_report(path, label=""):
    """Full pipeline evidence card — all modules active."""
    path = str(path)

    # M1 + M2 + M4 (M3 runs but is note-only)
    r = analyze_image(path, detector_ckpt=CKPT)

    # M5 — only when inconclusive (saves API calls)
    web = None
    if r.ai_verdict.value == "inconclusive":
        web = trace_provenance(path, detector_ckpt=CKPT)
        if web.get("upstream_verdict") in ("verified", "likely"):
            r.notes.append(
                f"M5: upstream → {web['upstream_verdict']} "
                f"(via {web['upstream_url']})"
            )
            r.notes.extend(web["notes"])
            r.ai_verdict = Verdict(web["upstream_verdict"])
            r.confidence = (
                web["chain"][-1]["confidence"]
                if web["chain"] else 0.75
            )
        elif web.get("ai_context_match"):
            r.notes.extend(web["notes"])
            r.ai_verdict = Verdict.LIKELY
            r.confidence = 0.7
        elif web.get("ai_gallery_match"):
            r.notes.append(
                "M5: found on AI gallery domains "
                f"({', '.join(web['domains'][:3])})"
            )

    # Print the evidence card
    p = r.evidence.provenance
    d = r.evidence.detector
    print(f"┌─ {label or Path(path).name}")
    print(f"│")
    print(f"│ VERDICT: {r.ai_verdict.value} ({r.confidence})")
    print(f"│")
    # M1
    print(f"│ M1 provenance:")
    print(f"│   c2pa: present={p.c2pa_present}"
          f" sig_valid={p.c2pa_signature_valid}"
          f" gen={p.c2pa_generator}")
    if p.c2pa_actions:
        print(f"│   actions: {p.c2pa_actions[:3]}")
    if p.generation_params_tool:
        print(f"│   gen-params: {p.generation_params_tool}"
              f" model={p.generation_params_model}")
    if p.iptc_source_category:
        print(f"│   iptc: {p.iptc_digital_source_type}"
              f" → {p.iptc_source_category}")
    if p.camera_exif_present:
        print(f"│   camera-exif: {p.camera_exif_fields} fields")
    if not any([p.c2pa_present, p.generation_params_tool,
                p.iptc_source_category, p.ai_metadata_hits]):
        print(f"│   (no metadata signals)")
    # M2
    print(f"│ M2 watermarks:")
    for w in r.evidence.watermarks:
        if not w.applicable:
            continue
        status = "DETECTED" if w.detected else "not detected"
        detail = f" p={w.bit_accuracy}" if w.bit_accuracy else ""
        print(f"│   {w.scheme:<22} {status}{detail}")
    # M4
    print(f"│ M4 classifier:")
    if d and d.valid:
        print(f"│   p(fake)={d.p_calibrated}"
              f"  drift={d.robustness_drift}")
    else:
        reason = d.notes if d else "no checkpoint"
        print(f"│   (unavailable: {reason})")
    # M5
    print(f"│ M5 web provenance:")
    if web and web["searched"]:
        print(f"│   matches={web['matches_found']}"
              f"  labels={web.get('best_guess_labels', [])}")
        if web.get("ai_context_match"):
            print(f"│   AI context: ✓ ({len([n for n in web['notes'] if 'signal' in n])} signals)")
        if web.get("ai_gallery_match"):
            print(f"│   AI gallery domains found")
    elif web:
        print(f"│   (no matches)")
    else:
        print(f"│   (skipped — verdict already determined)")
    # Notes
    print(f"│")
    print(f"│ notes: {r.notes}")
    print(f"└")
    print()
    return r

## Test 1 — upload your images

Upload a mix to see every module's response:
- **OpenAI/ChatGPT PNG** → expect M1 verified (C2PA), M4 high p
- **Content Authenticity Firefly** → expect M2 TrustMark-P detected
- **Stripped/compressed copy** → expect M4 likely, maybe M5 context
- **Phone photo** → expect inconclusive or unlikely
- **Known viral AI image (Pope etc.)** → expect M5 context match

In [ ]:
from google.colab import files

print("Upload images to analyze (any format, any source):")
up = files.upload()
for name in sorted(up):
    full_report(name)

## Test 2 — the transport gauntlet (full pipeline)

One image through all 7 hops — all modules active on every hop.
The composite table showing which modules contribute at each stage.

In [ ]:
import shutil, subprocess
from google.colab import files
from PIL import Image

print("Upload the image to test through transports:")
up = files.upload()
orig = Path(next(iter(up)))
HOP = Path("/content/hops_final"); HOP.mkdir(exist_ok=True)
im = Image.open(orig).convert("RGB")

hops = {"0-original": orig}
p = HOP/"1-resave.jpg";     im.save(p, quality=92);  hops["1-resave-jpg"] = p
p = HOP/"2-screenshot.png"; im.save(p);               hops["2-screenshot"] = p
p = HOP/"3-messenger.jpg"
im.resize((im.width//2, im.height//2)).save(p, quality=70)
hops["3-messenger"] = p
p = HOP/("4-strip"+orig.suffix); shutil.copy(orig, p)
subprocess.run(
    ["exiftool", "-overwrite_original", "-all=", str(p)],
    capture_output=True,
)
hops["4-exiftool-strip"] = p
p = HOP/"5-crop.jpg"
im.crop((50, 50, im.width-50, im.height-50)).save(p, quality=92)
hops["5-crop"] = p
p = HOP/"6-pil-reencode.png"; im.save(p)
hops["6-pil-reencode"] = p

for name, path in hops.items():
    full_report(path, label=name)

## Summary — the thesis measured

The evidence card for each transport shows which modules fire:

| transport | M1 (C2PA) | M2 (TrustMark) | M4 (classifier) | M5 (web) |
|---|---|---|---|---|
| original | ✓ verified | ✓? | p=high | skipped |
| re-save | · | ✓? | p=high | skipped |
| screenshot | · | ✓? | p=high | skipped |
| messenger | · | · | p=high | skipped |
| strip | ✓ | ✓? | p=high | skipped |
| crop | · | ✓? | p=high | skipped |
| re-encode | · | ✓? | p=high | skipped |

If M4 keeps p>0.8 across all rows, it fills every gap M1 and M2 leave.
M5 fires only on `inconclusive` verdicts — so its column activates only
when M4 is also absent or uncertain.

**No single module covers everything. Their union does.** That's the
evidence-fusion thesis, measured across four independent signal families
and seven transport scenarios.